# QEvict on DeepSeek-V2-Lite-Chat: LongBench, 10% KV budget, batched like the SnapKV baseline (Colab)

Runs **QEvict at a 10% KV-cache budget** on **`deepseek-ai/DeepSeek-V2-Lite-Chat`** over the paper's Table 2 LongBench
tasks except NarrativeQA: Qasper, MultiFieldQA-en, HotpotQA, 2WikiMQA, MuSiQue, GovReport, QMSum, MultiNews, TREC,
TriviaQA, SAMSum (every example), **under the protocol and batching of the DeepSeek-V2 SnapKV / KVQuant baselines**
(DefensiveKV's batched kvpress script), so the rows compare.

* **Protocol (DefensiveKV, pinned commit):** its context / question templates and `max_new_tokens`, its kvpress
  `preprocess`, its `calculate_metrics`. The context is prefilled and compressed, the question is appended, the answer
  is decoded greedily. No chat template and an empty BOS for TREC / TriviaQA / SAMSum; SAMSum also stops at a newline.
* **Batching (the baseline's defaults):** longest contexts first, at most 8 rows and 140,000 padded prefill tokens per
  batch, no truncation. Contexts and question blocks are left-padded, positions are per row, all rows decode in
  lockstep, and each answer is cut at its first EOS.
* **QEvict (the paper's settings):** budget `0.10 x (context + question + max_new_tokens)` per row, W = R = 8, 5 sink
  tokens, 128 recent tokens, alpha = 0.70 of the byte budget to INT2, read ratio rho = 0.25, bidirectional F<->Q
  migration. Each row has its own cache: every non-attention layer runs on the padded batch, and attention is split
  by row, so a row's cache only ever holds its own tokens. The question block is fed one column at a time (the cache
  takes one prompt, then single-token steps through its gated read).

Harness: `modules/evaluation/longbench_qevict_batched.py`. Needs an **A100 80GB**. Results go to Google Drive, one
file per task; rerun all cells to resume (finished tasks are skipped).

In [ ]:
# ===================== Settings =====================
BRANCH         = "Clustered_int2_paused_fast_deepseek"
REPO_URL       = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT     = "7595a60c72750637874a717625b00a1a11e46c7f"
DKV_URL        = "https://github.com/FFY0/DefensiveKV.git"
DKV_COMMIT     = "9251b95e7e282a6c18dd98ef6318dd489e639a58"
MODEL_ID       = "deepseek-ai/DeepSeek-V2-Lite-Chat"
MODEL_REVISION = "85864749cd611b4353ce1decdb286193298f64c7"
MODEL_DIR      = "/content/models/DeepSeek-V2-Lite-Chat"
DRIVE_ROOT     = "/content/drive/MyDrive/qevict_deepseek_v2_lite_longbench_b010_batched"

TASKS = ["qasper", "multifieldqa_en", "hotpotqa", "2wikimqa", "musique",
         "gov_report", "qmsum", "multi_news", "trec", "triviaqa", "samsum"]
LABEL = {"qasper": "Qasper", "multifieldqa_en": "Mul.QA", "hotpotqa": "Hot.QA", "2wikimqa": "2Wi.QA",
         "musique": "Musique", "gov_report": "Gov.Re.", "qmsum": "QMSum", "multi_news": "M.News",
         "trec": "TREC", "triviaqa": "Tri.QA", "samsum": "SAMSum"}
# The baseline script's batching arguments, and the method's budget.
RUN = {"cache_budget": 0.10, "batch_size": 8, "max_batch_tokens": 140000, "max_prompt_tokens": 0,
       "config": "configs/longbench_deepseek_v2_lite_flash.yaml"}

In [ ]:
import subprocess, json, os, sys, shutil, time, zipfile, statistics
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0].split(",")
print("GPU:", gpu[0].strip(), f"{float(gpu[1]) / 1024:.0f} GB")
if float(gpu[1]) / 1024 < 75:
    raise RuntimeError("This needs an 80 GB GPU.")

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

def checkout(url, path, commit, branch=None):
    path = Path(path)
    if not path.exists():
        subprocess.run(["git", "clone", *(["--branch", branch] if branch else []), url, str(path)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", *([branch] if branch else [])], cwd=path, check=True)
    subprocess.run(["git", "checkout", "-q", "--force", commit], cwd=path, check=True)
    print(path.name, subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=path, text=True).strip())
    return path

REPO = checkout(REPO_URL, "/content/RecoverKv", PIN_COMMIT, BRANCH)
DKV = checkout(DKV_URL, "/content/DefensiveKV", DKV_COMMIT)

## Install the pinned stack

A Python 3.12 venv with the stack the recorded GPU runs used: torch 2.6.0 (+cu124; the eviction's `torch.compile`
fails on 2.7.1), transformers 4.47.1 (the cache refuses newer), and a prebuilt flash-attn wheel that loads against
torch 2.6.0 (2.7.4.post1; the 2.8.x wheels for torch 2.6 fail on a C++ ABI symbol).

In [ ]:
VENV = Path("/content/py312_env")
PY = str(VENV / "bin" / "python")
PINS = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0", "triton": "3.2.0",
        "transformers": "4.47.1"}
STACK_ID = json.dumps({"python": "3.12", "cuda": "cu124", **PINS}, sort_keys=True)
MARK = VENV / ".recoverkv_stack"
_DROP = ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH",
         "PIP_CONSTRAINT", "MPLBACKEND")
VENV_ENV = {k: v for k, v in os.environ.items() if k not in _DROP}
VENV_ENV.update(VIRTUAL_ENV=str(VENV), PATH=f"{VENV / 'bin'}:{os.environ.get('PATH', '')}", MPLBACKEND="Agg")

def sh(*cmd, env=None):
    subprocess.run([str(c) for c in cmd], check=True, env=VENV_ENV if env is None else env)

def vpip(*a):
    sh(PY, "-m", "pip", "install", "-q", "--disable-pip-version-check", *a)

def versions():
    code = ("import json, importlib.metadata as m\nout = {}\n"
            "for d in ('torch','torchvision','torchaudio','triton','transformers','flash_attn'):\n"
            "    try: out[d] = m.version(d)\n    except m.PackageNotFoundError: out[d] = None\n"
            "print(json.dumps(out))")
    v = json.loads(subprocess.check_output([PY, "-c", code], env=VENV_ENV, text=True))
    bad = {k: v.get(k) for k, want in PINS.items() if (v.get(k) or "").split("+")[0] != want}
    if bad:
        raise RuntimeError(f"the venv drifted from the pinned stack: {bad}. Delete {VENV} and rerun.")
    return v

def fa_loads():
    return subprocess.run([PY, "-c", "import torch, flash_attn_2_cuda, flash_attn"], env=VENV_ENV,
                          capture_output=True).returncode == 0

if not (MARK.exists() and MARK.read_text() == STACK_ID and Path(PY).exists()):
    shutil.rmtree(VENV, ignore_errors=True)
    if not (shutil.which("python3.12") and subprocess.run(["python3.12", "-c", "import ensurepip, venv"],
                                                          capture_output=True).returncode == 0):
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        sh("apt-get", "update", "-qq", env=apt_env)
        sh("apt-get", "install", "-y", "-qq", "python3.12", "python3.12-venv", "python3.12-dev", env=apt_env)
    sh("python3.12", "-m", "venv", VENV, env=os.environ)
    constraints = VENV / "constraints.txt"
    constraints.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
    vpip("--upgrade", "pip")
    vpip("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
         "--index-url", "https://download.pytorch.org/whl/cu124")
    vpip("-c", constraints,
         "transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
         "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm",
         "pandas", "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba")
    versions()
    tv, abi, py = subprocess.check_output([PY, "-c",
        "import torch, sys; print(torch.__version__.split('+')[0]); "
        "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
        "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], env=VENV_ENV, text=True).split()
    tmm = ".".join(tv.split(".")[:2])
    for v in ("2.7.4.post1", "2.8.3", "2.8.2"):
        u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
             f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
        try:
            vpip("--no-deps", "-c", constraints, u)
        except subprocess.CalledProcessError:
            continue
        if fa_loads():
            print("flash-attn wheel:", u.rsplit("/", 1)[1])
            break
        sh(PY, "-m", "pip", "uninstall", "-y", "-q", "flash-attn")
    else:
        raise RuntimeError(f"no prebuilt flash-attn wheel loads against torch {tv} / {py} / cxx11abi={abi}")
    MARK.write_text(STACK_ID)
print("venv:", versions())

## Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir=MODEL_DIR,
                  allow_patterns=["*.json", "*.safetensors", "*.py", "tokenizer*"])
# DeepSeek-V2-Lite-Chat at the pinned revision: the remote code the repo's MLA forward was checked against, and the
# chat template the protocol wraps the prompts in.
import hashlib
for f in ("config.json", "configuration_deepseek.py", "modeling_deepseek.py"):
    h = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    assert h(Path(MODEL_DIR) / f) == h(REPO / "models/deepseek_v2/upstream" / f), f"{f}: not DeepSeek-V2-Lite"
assert json.loads((Path(MODEL_DIR) / "tokenizer_config.json").read_text()).get("chat_template"), "no chat template"
print("model:", MODEL_ID, MODEL_REVISION[:12])

LB_DIR = Path("/content/longbench")
LB_DIR.mkdir(parents=True, exist_ok=True)
if not all((LB_DIR / f"{t}.jsonl").exists() for t in TASKS):
    with zipfile.ZipFile(hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset")) as z:
        for t in TASKS:
            member = next(n for n in z.namelist() if n.endswith(f"/{t}.jsonl") or n == f"{t}.jsonl")
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
N = {t: sum(1 for l in open(LB_DIR / f"{t}.jsonl") if l.strip()) for t in TASKS}
print(N)

ENV = dict(VENV_ENV, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8", TOKENIZERS_PARALLELISM="false",
           HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")

## Run

One process per task (as the baseline script runs). A task is finished when its `<task>_qevict_b0.1_bs8.meta.json` is
on Drive; its sidecar must report `read_gate: gated` and every example.

In [ ]:
OUT = ROOT / "qevict_b010"
OUT.mkdir(parents=True, exist_ok=True)
STEM = "{task}_qevict_b%s_bs%d" % (RUN["cache_budget"], RUN["batch_size"]) + (
    f"_trunc{RUN['max_prompt_tokens']}" if RUN["max_prompt_tokens"] else "")
EXPECT_GATE = "gated"

for t in TASKS:
    meta_path = OUT / (STEM.format(task=t) + ".meta.json")
    if not meta_path.exists():
        cmd = [PY, "-m", "modules.evaluation.longbench_qevict_batched", "--model", MODEL_DIR,
               "--revision", MODEL_REVISION, "--task", t, "--data_dir", str(LB_DIR), "--defensivekv_dir", str(DKV),
               "--config", RUN["config"], "--cache_budget", str(RUN["cache_budget"]),
               "--batch_size", str(RUN["batch_size"]), "--max_batch_tokens", str(RUN["max_batch_tokens"]),
               "--max_prompt_tokens", str(RUN["max_prompt_tokens"]), "--out_dir", str(OUT)]
        print(f"=== {t}")
        with open(ROOT / "run.log", "a", buffering=1) as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1)
            for line in p.stdout:
                log.write(line)
                if line.startswith(("batch ", "read gate")) or "Error" in line or "Traceback" in line:
                    print("   ", line.rstrip()[:200])
            if p.wait() != 0:
                raise RuntimeError(f"{t} stopped; see {ROOT / 'run.log'}. Rerun this cell to resume.")
    meta = json.loads(meta_path.read_text())
    assert meta["read_gate"]["verdict"] == EXPECT_GATE and meta["n"] == N[t], (t, meta["read_gate"], meta["n"])
    print(f"{t:16s} {meta['n']:4d} examples  score {meta['score']:6.2f}  read_gate={meta['read_gate']['verdict']}  "
          f"{meta['n_batches']} batches  {meta['seconds'] / 60:.0f} min  peak {meta['peak_gb']} GB")

## Scores

In [ ]:
scores = [json.loads((OUT / (STEM.format(task=t) + ".meta.json")).read_text())["score"] for t in TASKS]
row = ["QEvict", "10%", *[f"{x:.2f}" for x in scores], f"{statistics.mean(scores):.2f}"]
head = ["Method", "KV budget", *[LABEL[t] for t in TASKS], "Avg (11)"]
table = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head), "| " + " | ".join(row) + " |"]
(ROOT / "table_b010.md").write_text("\n".join(table) + "\n")
print("\n".join(table))
print("\nLaTeX:\n" + " & ".join(row).replace("%", r"\%") + r" \\")